# Magnetic Resonance Imaging II
 
## Exercise 2
 
---
 
**Prof. Dr. Markus Zimmermann**
 
**Mohamed Wassim Bensalem**
 
---

In Exercise 1 you built a single-channel GRE sequence and reconstructed images using a direct inverse Fourier transform. This notebook extends that pipeline in two directions. First, the acquisition is accelerated by sampling only every second phase-encoding line — a strategy called **undersampling**. Second, the scanner is equipped with an array of 8 receiver coils, each of which records the signal with a different spatial sensitivity profile. Together, these changes mean that a simple inverse Fourier transform is no longer sufficient: the missing k-space lines and the multi-channel structure must both be handled explicitly in the reconstruction.
 
To do this in a general and extensible way, you will formulate the reconstruction as a **linear inverse problem** and solve it iteratively using the **Conjugate Gradient (CG)** method. The encoding process — coil sensitivities, Fourier transform, and k-space sampling mask — is expressed as a composition of **linear operators**, and the reconstruction amounts to inverting this composition. This operator formalism is the foundation of virtually all modern advanced MRI reconstruction methods, from compressed sensing to deep learning-based approaches, so understanding it here will carry forward throughout the course.
 
You will use the same libraries as in Exercise 1, plus two additional modules:
 
- **`mri.parim.analytic`** — provides `cyclic_head_coil`, a function that generates analytic coil sensitivity maps for a circular head coil array.
- **`mri.conopt.linops`** — provides `LinearOperator`, a base class for defining linear operators that support composition via `@` and adjoint via `.H`.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
from tqdm import tqdm

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(1, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.linops import LinearOperator
from mri.parim.analytic import cyclic_head_coil

## System and Sequence Parameters
 
The parameters are identical to Exercise 1, with one addition: `n_channels` defines the number of receiver coils in the array. `n_x` and `n_y` define the full k-space matrix size — note that the accelerated acquisition will only acquire `n_y // 2` of the $n_y$ phase-encoding lines, but the reconstruction target remains an $n_x \times n_y$ image.

In [ ]:
seq_filename = "gre_accelerated.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
n_channels = 8
R = 1

case = ["T1-weighted", "T2-weighted", "PD-weighted"][2]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

torch.set_default_device("cpu")

## System and Sequence Object

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Sequence Components
 
The sequence building blocks are identical to Exercise 1. They are provided here so you can focus on the new reconstruction content.

In [ ]:
delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=gx.flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

te_delay = (
    np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)
tr_delay = (
    np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

## Task 1: Undersampled Sequence Assembly

In Exercise 1 you acquired all $n_y = 128$ phase-encoding lines, uniformly covering k-space from $-k_{y,\text{max}}$ to $+k_{y,\text{max}}$. Here you will acquire only every second line, reducing the scan time by a factor of 2. This is called **undersampling** with an **acceleration factor** $R = 2$.
 
The only change from Exercise 1 is in how `phase_areas` is computed. Instead of `num=n_y` steps, you will use `num=n_y // 2` steps — but the start and end of the range stay the same. This means the spacing between acquired lines doubles, leaving every other line empty.
 
**Your task:** Assemble the GRE sequence with undersampled phase encoding. Copy your sequence loop from Exercise 1 and change the `num` argument in `np.linspace()` from `n_y` to `n_y // 2`. Everything else stays the same.

In [ ]:
# Your solution here




## Verification and Export

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot(time_range=(0.0, tr - tr_delay + te_delay))

seq.write(seq_filename, v141_compat=True)

## Phantom and Multi-Channel Simulation

In Exercise 1, the phantom had a single receive channel — equivalent to a perfectly uniform body coil. Here, `n_channels = 8` receiver coils are placed around the head, each with a different spatial sensitivity profile. The function `cyclic_head_coil()` returns a tensor of shape `(n_channels, n_x, n_y, 1)` containing the complex-valued sensitivity map for each coil. Assigning this to `phantom.coil_sens` tells MRzero to simulate the signal as it would appear on each coil separately.
 
The output `signal` now has shape $(n_y/2 \cdot n_x,\ n_\text{channels})$ — for each of the acquired k-space samples, there is one complex value per coil.

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

coil_sensitivities_recon = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon[:, :, :, 0].flip((1, 2))

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
    fov_scaling=(phantom.size[0] / fov_x, phantom.size[1] / fov_y, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Task 2: Coil Images

Before building any operators, it is worth looking at what each coil actually sees. You will reshape the raw signal into a per-coil k-space, zero-fill the missing lines, and apply a simple inverse Fourier transform to each coil independently — the same reconstruction you used in Exercise 1, but applied channel by channel.
 
The raw signal has shape $(n_y/2 \cdot n_x,\ n_\text{channels})$. Reshape it to $(n_y/2,\ n_x,\ n_\text{channels})$ and then permute the axes to $(n_\text{channels},\ n_x,\ n_y/2)$ so that each coil's data is a 2D array indexed as `[channel, kx, ky]`.
 
To zero-fill, create a tensor `data_full` of shape $(n_\text{channels},\ n_x,\ n_y)$ filled with zeros, and copy the acquired lines into the even-indexed positions along the last axis (`::2`). This places the acquired k-space lines back into their correct positions in the full k-space grid, leaving the unacquired lines as zero.

Then reconstruct each coil image by applying the 2D inverse Fourier transform along the spatial axes $(1, 2)$:
 
$$\text{ifftshift} \;\rightarrow\; \text{ifftn} \;\rightarrow\; \text{fftshift}$$
 
**Your tasks:**
 
1. Reshape and permute `signal` to obtain `data` of shape $(n_\text{channels},\ n_x,\ n_y/2)$.
2. Create `data_full` of shape $(n_\text{channels},\ n_x,\ n_y)$ and place the acquired data into `data_full[:, :, ::2]`.
3. Reconstruct `coil_images` by applying the inverse Fourier transform along axes `(1, 2)`.
4. Display the magnitude of all 8 coil images in a 2×4 grid.

Look at the images carefully. Why do the coil images look different from each other? What determines the spatial variation in signal intensity within each coil image?

Display the magnitude of `coil_sensitivities_recon` alongside the coil images to verify that the spatial sensitivity profiles match the signal intensity patterns you observe.

In [ ]:
# Your solution here




## Task 3: The Fourier Transform Operator

You are now going to build the operator formalism that will allow you to express the full parallel imaging forward model compactly and solve the reconstruction problem iteratively.
 
All operators in this exercise derive from `LinearOperator`, a base class that provides operator composition via `@` and the adjoint via `.H`. To define a new operator, you subclass `LinearOperator` and implement two methods:
 
- `_matvec(self, x)` — the **forward** operation: given a flat input vector $\mathbf{x}$, apply the operator and return a flat output vector.
- `_rmatvec(self, x)` — the **adjoint** operation: given a flat input vector, apply the Hermitian transpose of the operator and return a flat output vector.

Both methods receive and must return **flat (1D) real-valued tensors** of `dtype=torch.float32`. Since MRI signals are complex-valued, complex tensors are represented internally as real tensors of twice the size by reinterpreting the memory: `.view(torch.float32)` converts a complex tensor to its real representation, and `.view(torch.complex64)` converts it back. The constructor must set `self.dtype = torch.float32` and `self.shape = (output_size, input_size)` where the sizes count real-valued elements — i.e. twice the number of complex elements. The working shapes for the forward and adjoint passes are stored as `self.forward_shape` and `self.adjoint_shape`.
 
The first operator to implement is the **Fourier Transform Operator** $\mathbf{F}$, which maps a multi-channel image to multi-channel k-space. Both the input and output have shape $(n_\text{channels},\ n_x,\ n_y)$, so $\mathbf{F}$ is a square operator.
 
The MRI Fourier convention used throughout this course defines the transforms as:
 
$$\mathbf{F}:\quad \text{ifftshift} \;\rightarrow\; \text{fftn} \;\rightarrow\; \text{fftshift}$$
$$\mathbf{F}^\mathrm{H}:\quad \text{ifftshift} \;\rightarrow\; \text{ifftn} \;\rightarrow\; \text{fftshift}$$
 
applied along the spatial axes only (axes 1 and 2 for a $(n_\text{channels},\ n_x,\ n_y)$ tensor). The `ifftshift` before the transform and `fftshift` after handle the DC-centered convention used in MRI. The forward and adjoint differ only in the direction of the transform (`fftn` vs `ifftn`). To ensure $\mathbf{F}^\mathrm{H} \mathbf{F} = \mathbf{I}$, both must use `norm="ortho"`.
 
**Your task:** Implement `FourierTransformOperator`. The `axes` parameter specifies which dimensions to transform along — store it and use it in both `_matvec` and `_rmatvec`.

In [ ]:
# Your solution here




## Task 4: Channel and Mask Operators

With the Fourier operator in place, you now need two more operators to complete the forward model.
 
---
 
**Channel Operator $\mathbf{C}$** maps a single-channel image of shape $(1,\ n_x,\ n_y)$ to a multi-channel image of shape $(n_\text{channels},\ n_x,\ n_y)$ by multiplying pointwise with the coil sensitivity maps $s_c$:
 
$$\mathbf{C}:\quad y_c = s_c \cdot x \qquad \text{for each coil } c$$
$$\mathbf{C}^\mathrm{H}:\quad y = \sum_c s_c^* \cdot x_c$$
 
The adjoint sums the coil signals weighted by the conjugate sensitivities, collapsing the channel dimension back to a single image. Note that the forward and adjoint shapes differ: $\mathbf{C}$ maps $(1,\ n_x,\ n_y) \rightarrow (n_\text{channels},\ n_x,\ n_y)$, so it is not a square operator and `self.shape` must reflect this asymmetry.
 
**Diagonal (Mask) Operator $\mathbf{M}$** applies the k-space sampling mask. It has shape $(n_\text{channels},\ n_x,\ n_y)$ on both input and output, and simply multiplies each element of k-space pointwise by the binary mask — setting unacquired lines to zero. Since the mask is real-valued and binary, the adjoint is identical to the forward: $\mathbf{M}^\mathrm{H} = \mathbf{M}$.
 
**Your tasks:**
 
1. Implement `ChannelOperator`. The constructor takes `data_shape = (n_x, n_y)` and derives `n_channels` from `coil_sensitivities.shape[0]`. Set `self.shape` to reflect the non-square mapping from $(1,\ n_x,\ n_y)$ to $(n_\text{channels},\ n_x,\ n_y)$.
2. Implement `DiagonalOperator`. The constructor takes `data_shape = (n_channels, n_x, n_y)`. Both `_matvec` and `_rmatvec` apply the same pointwise multiply.

In [ ]:
# Your solution here




## Task 5: Compose the Forward Operator and Compute $A^H d$

You now have all three building blocks. The full parallel imaging forward model — mapping a single-channel image to undersampled multi-channel k-space — is:
 
$$\mathbf{A} = \mathbf{M} \mathbf{F} \mathbf{C}$$
 
where:
 
- $\mathbf{C}$ applies coil sensitivities: $(1,\ n_x,\ n_y) \rightarrow (n_\text{channels},\ n_x,\ n_y)$
- $\mathbf{F}$ applies the Fourier transform: $(n_\text{channels},\ n_x,\ n_y) \rightarrow (n_\text{channels},\ n_x,\ n_y)$
- $\mathbf{M}$ applies the sampling mask: $(n_\text{channels},\ n_x,\ n_y) \rightarrow (n_\text{channels},\ n_x,\ n_y)$

In code, `A = M @ F @ C` composes these left to right, so `A @ x` computes $\mathbf{M}(\mathbf{F}(\mathbf{C}\mathbf{x}))$.
 
The acquired data `data_full` has shape $(n_\text{channels},\ n_x,\ n_y)$ with zeros at the unacquired lines. Reinterpreting it as a real vector with `.view(torch.float32).ravel()` gives the measurement vector $\mathbf{d}$.
 
The vector $\mathbf{A}^\mathrm{H} \mathbf{d}$ is the **matched filter output** — it applies the adjoint of the full encoding to the measurements, mapping them back into image space. It is the starting point for iterative reconstruction, and already contains most of the image information despite the aliasing artifacts from undersampling.
 
**Your tasks:**
 
1. Construct the sampling mask `mask` of shape $(n_\text{channels},\ n_x,\ n_y)$ with ones at even-indexed k-space lines (`::2`) and zeros elsewhere. Use `dtype=torch.complex64`.
2. Instantiate $\mathbf{F}$, $\mathbf{C}$, and $\mathbf{M}$ using your operator classes. Pass `coil_sensitivities_recon` to `ChannelOperator`.
3. Compose `A = M @ F @ C`.
4. Compute `AHd = A.H @ data_full.view(torch.float32).ravel()`.
5. Reinterpret `AHd` as complex with `.view(torch.complex64)`, reshape to $(n_x,\ n_y)$, and display its magnitude.

What artifact do you see in the image? Where does it come from?

In [ ]:
# Your solution here




## Task 6: Iterative Reconstruction with Conjugate Gradient
 
The zero-filled reconstruction $\mathbf{A}^\mathrm{H} \mathbf{d}$ shows the aliasing artifact caused by the missing k-space lines. To recover the true image, you need to solve the **normal equations**:
 
$$\mathbf{Q}\, \mathbf{x} = \mathbf{b} \qquad \text{where} \quad \mathbf{Q} = \mathbf{A}^\mathrm{H} \mathbf{A}, \quad \mathbf{b} = \mathbf{A}^\mathrm{H} \mathbf{d}$$
 
$\mathbf{Q}$ is Hermitian positive semi-definite. The **Conjugate Gradient (CG)** method solves such systems iteratively without ever forming $\mathbf{Q}$ explicitly — it only requires the ability to evaluate $\mathbf{Q} \mathbf{x}$ for any vector $\mathbf{x}$, which is exactly what the operator formalism provides.
 
The CG solver is provided below. It takes the operator $\mathbf{Q}$, the right-hand side $\mathbf{b} = \mathbf{A}^\mathrm{H} \mathbf{d}$, and an initial guess $\mathbf{x}$, and returns the reconstructed image vector after `maxiter` iterations.

In [ ]:
def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter)):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

**Your tasks:**
 
1. Construct $\mathbf{Q} = \mathbf{A}^\mathrm{H} \mathbf{A}$ using `A.H @ A`.
2. Set $\mathbf{b} = \mathbf{A}^\mathrm{H} \mathbf{d}$ (already computed in Task 5).
3. Initialize $\mathbf{x}$ as a zero tensor of shape $(2 \cdot n_x \cdot n_y,)$ with `dtype=torch.float32` (real representation of the complex image).
4. Call `conjugate_gradient(Q, b, x, maxiter=30)` and store the result in `x_recon`.
5. Reinterpret `x_recon` as complex with `.view(torch.complex64)`, reshape to $(n_x,\ n_y)$, and display its magnitude side by side with `AHd_image`. What has the CG reconstruction recovered that the zero-filled reconstruction could not?

In [ ]:
# Your solution here


